Use PhoBert to classify the financial report.

In [4]:
import pandas as pd

# Vì file nằm ngay trong thư mục gốc của Sidebar
news_df = pd.read_csv('/content/news_all.csv')

# Kiểm tra dữ liệu
print(f"Tổng số tin tức: {len(news_df)}")
print(news_df.head())

Tổng số tin tức: 1770
  ticker                                              title  \
0    VIC  VN-Index tăng mạnh nhờ nhóm cổ phiếu Vingroup ...   
1    VIC  Vingroup trở thành tập đoàn lớn thứ 4 Đông Nam...   
2    VHM  Thị trường chứng khoán tăng hơn 22 điểm - Viet...   
3    VIC  Chứng khoán Việt tăng mạnh top đầu châu Á, tài...   
4    VIC  Vingroup là doanh nghiệp có vốn hóa lớn thứ 4 ...   

                                                 url                 date  \
0  https://news.google.com/rss/articles/CBMiqAFBV...  2026-04-28 17:27:00   
1  https://news.google.com/rss/articles/CBMiigFBV...  2026-04-28 17:15:18   
2  https://news.google.com/rss/articles/CBMib0FVX...  2026-04-28 17:14:29   
3  https://news.google.com/rss/articles/CBMirwFBV...  2026-04-28 17:05:10   
4  https://news.google.com/rss/articles/CBMilAFBV...  2026-04-28 17:02:28   

                      source  \
0  thuonghieucongluan.com.vn   
1                      Znews   
2                 Vietnam.vn   
3     Bá

In [5]:
!pip install pyvi

from pyvi import ViTokenizer

def segment_text(text):
    if isinstance(text, str):
        return ViTokenizer.tokenize(text)
    return ""

# Áp dụng tách từ cho cột tiêu đề tin tức
news_df['title_segmented'] = news_df['title'].apply(segment_text)

print("Ví dụ sau khi tách từ:")
print(news_df['title_segmented'].iloc[0])
# Kết quả sẽ có dạng: "giá cổ_phiếu fpt tăng_trưởng mạnh"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.1 MB/s eta 0:00:00
Ví dụ sau khi tách từ:
VN - Index tăng mạnh nhờ nhóm cổ_phiếu Vingroup nổi sóng lớn - thuonghieucongluan . com . vn


In [7]:
# Cleaning

import re

def clean_vn_finance_text(text):
    if not isinstance(text, str):
        return ""

    # 1. Xóa tên miền và nguồn tin thường xuất hiện sau dấu gạch ngang cuối cùng
    # Ví dụ: "Giá vàng tăng cao - CafeF" -> "Giá vàng tăng cao"
    text = re.split(r'\s-\s|\s\|\s', text)[0]

    # 2. Xóa các website phổ biến nếu còn sót lại
    text = re.sub(r'\s[\w\.]+\.(com|vn|net|org)\S*', '', text)

    # 3. Xóa các ký tự đặc biệt thừa thãi (nhưng giữ lại dấu % và số vì quan trọng trong tài chính)
    text = re.sub(r'[^\w\s\d%.,]', ' ', text)

    # 4. Xóa khoảng trắng thừa
    text = ' '.join(text.split())

    return text

# Áp dụng làm sạch trước khi tách từ
news_df['title_clean'] = news_df['title'].apply(clean_vn_finance_text)

# Sau đó mới thực hiện tách từ (Segmentation)
news_df['title_segmented'] = news_df['title_clean'].apply(lambda x: ViTokenizer.tokenize(x).lower())

print("Sau khi làm sạch và tách từ:")
print(news_df['title_segmented'].iloc[0])

Sau khi làm sạch và tách từ:
vn index tăng mạnh nhờ nhóm cổ_phiếu vingroup nổi sóng lớn


In [6]:
# Sau khi đã có nhãn (giả sử tên cột là 'sentiment')
# Hãy mount drive để lưu bền vững
from google.colab import drive
drive.mount('/content/drive')

# Lưu file kết quả vào Drive
news_df.to_csv('/content/drive/MyDrive/news_with_sentiment.csv', index=False, encoding='utf-8-sig')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
from transformers import pipeline

# Khởi tạo pipeline phân loại cảm xúc với PhoBERT đã fine-tune
# Mô hình này trả về nhãn: POS (Tích cực), NEG (Tiêu cực), NEU (Trung lập)
sentiment_model = pipeline("sentiment-analysis", model="wonrax/phobert-base-vietnamese-sentiment")

def get_sentiment(text):
    if not text: return 0
    # Cắt ngắn văn bản nếu quá dài (PhoBERT tối đa 256 tokens cho bản này)
    result = sentiment_model(text[:250])[0]

    # Chuyển đổi nhãn thành con số để tính toán sau này
    # POS -> 1, NEG -> -1, NEU -> 0
    label = result['label']
    score = result['score']

    if label == 'NEG':
        return -score
    elif label == 'POS':
        return score
    return 0

# Test thử
print(get_sentiment("lợi nhuận fpt tăng trưởng kỷ lục"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/17.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

0


In [11]:
# Thử dùng mô hình này (được đánh giá rất cao cho Sentiment tiếng Việt)
model_name = "bmd1905/vietnamese-sentiment-visobert"
# Hoặc giữ nguyên wonrax nhưng thêm config xử lý lỗi

sentiment_model = pipeline(
    "sentiment-analysis",
    model="wonrax/phobert-base-vietnamese-sentiment",
    device=0 # Dùng GPU để chạy nhanh hơn
)

# Thử lại với hàm xử lý text sạch hơn
test_text = "Lợi nhuận FPT tăng trưởng kỷ lục"
result = sentiment_model(test_text)
print(result)

test = "FPT báo lỗ nặng"
result1 = sentiment_model(test)
print(result1)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: wonrax/phobert-base-vietnamese-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'label': 'POS', 'score': 0.5537270903587341}]
[{'label': 'NEG', 'score': 0.9852229356765747}]


In [ ]:
# Lưu vào Drive để dùng cho bước xây dựng model GRU/Bi-GRU dự báo giá
news_df.to_csv('/content/drive/MyDrive/processed_finance_news_with_sentiment.csv', index=False, encoding='utf-8-sig')

Manual Sentiment Classification combined with PhoBERT

In [14]:
def get_final_score(result):
    label = result[0]['label']
    score = result[0]['score']

    # Nếu Tiêu cực, trả về số âm
    if label == 'NEG':
        return -score
    # Nếu Tích cực, trả về số dương
    elif label == 'POS':
        return score
    # Trung lập
    return 0

# Test lại
print(get_final_score(sentiment_model("Lợi nhuận FPT tăng trưởng kỷ lục"))) # Sẽ ra ~0.55

0.5537270903587341


In [12]:
FINANCE_LEXICON = {
    "tích_cực": ["tăng_trưởng", "vượt_kế_hoạch", "lãi", "kỷ_lục", "bứt_phá", "hấp_dẫn", "trần", "khả_quan"],
    "tiêu_cực": ["lỗ", "sụt_giảm", "áp_lực", "cảnh_báo", "hủy_niêm_yết", "bán_tháo", "giảm_sàn", "nợ", "vỡ_nợ"]
}

def manual_feature_score(text_segmented):
    # Trả về điểm thưởng dựa trên sự xuất hiện của từ khóa
    score = 0
    for word in FINANCE_LEXICON["tích_cực"]:
        if word in text_segmented: score += 0.3
    for word in FINANCE_LEXICON["tiêu_cực"]:
        if word in text_segmented: score -= 0.3
    return score

In [15]:
def get_hybrid_sentiment(text_segmented):
    # 1. Lấy điểm từ AI (ViSoBERT)
    res = sentiment_model(text_segmented[:250])
    ai_score = get_final_score(res) # Hàm get_final_score đã viết ở trên

    # 2. Lấy điểm từ Manual Feature
    manual_score = manual_feature_score(text_segmented)

    # 3. Kết hợp (Trọng số có thể điều chỉnh, ví dụ 70% AI - 30% Manual)
    final_score = ai_score + manual_score

    # Giới hạn trong khoảng [-1, 1]
    return max(min(final_score, 1.0), -1.0)

# Áp dụng cho DataFrame
news_df['final_sentiment'] = news_df['title_segmented'].progress_apply(get_hybrid_sentiment)

100%|██████████| 1770/1770 [03:30<00:00,  8.39it/s]


In [17]:
import pandas as pd
import numpy as np

# 1. Định nghĩa bộ từ điển (Bạn có thể mở rộng thêm tùy ý)
FINANCE_LEXICON = {
    "tích_cực": ["tăng_trưởng", "vượt_kế_hoạch", "lãi", "kỷ_lục", "bứt_phá", "hấp_dẫn", "trần", "khả_quan", "tăng_mạnh", "hợp_tác"],
    "tiêu_cực": ["lỗ", "sụt_giảm", "áp_lực", "cảnh_báo", "hủy_niêm_yết", "bán_tháo", "giảm_sàn", "nợ", "vỡ_nợ", "giảm_sâu"]
}

def manual_feature_score(text_segmented):
    score = 0
    found_pos = [word for word in FINANCE_LEXICON["tích_cực"] if word in text_segmented]
    found_neg = [word for word in FINANCE_LEXICON["tiêu_cực"] if word in text_segmented]

    # Mỗi từ khóa xuất hiện cộng/trừ 0.25 điểm
    score += len(found_pos) * 0.25
    score -= len(found_neg) * 0.25

    return score, found_pos, found_neg

def test_hybrid_sentiment(raw_title):
    # Bước 1: Tiền xử lý & Tách từ
    segmented = ViTokenizer.tokenize(raw_title).lower()

    # Bước 2: AI Score (ViSoBERT)
    ai_res = sentiment_model(raw_title[:250])[0]
    ai_label = ai_res['label']
    ai_confidence = ai_res['score']

    # Chuyển nhãn AI thành điểm số
    ai_base_score = ai_confidence if ai_label == 'POS' else (-ai_confidence if ai_label == 'NEG' else 0)

    # Bước 3: Manual Score
    m_score, pos_words, neg_words = manual_feature_score(segmented)

    # Bước 4: Tính tổng Hybrid
    final_score = np.clip(ai_base_score + m_score, -1.0, 1.0)

    # In kết quả test
    print(f"--- TEST SENTIMENT ---")
    print(f"Tiêu đề: {raw_title}")
    print(f"Tách từ: {segmented}")
    print(f"AI Predict: {ai_label} ({ai_confidence:.2f})")
    print(f"Manual Keywords: POS: {pos_words} | NEG: {neg_words}")
    print(f"Manual Score: {m_score:+.2f}")
    print(f"==> FINAL HYBRID SCORE: {final_score:+.2f}")
    print("-" * 30)

# --- CHẠY THỬ CÁC KỊCH BẢN ---
test_titles = [
    "FPT báo lãi kỷ lục trong quý 1 năm 2026",
    "Áp lực bán tháo khiến thị trường giảm sàn hàng loạt",
    "Mặc dù gặp khó khăn, doanh nghiệp vẫn vượt kế hoạch lợi nhuận",
    "Cảnh báo nợ xấu tăng cao tại các ngân hàng"
]

for title in test_titles:
    test_hybrid_sentiment(title)

--- TEST SENTIMENT ---
Tiêu đề: FPT báo lãi kỷ lục trong quý 1 năm 2026
Tách từ: fpt báo lãi kỷ_lục trong quý 1 năm 2026
AI Predict: POS (0.69)
Manual Keywords: POS: ['lãi', 'kỷ_lục'] | NEG: []
Manual Score: +0.50
==> FINAL HYBRID SCORE: +1.00
------------------------------
--- TEST SENTIMENT ---
Tiêu đề: Áp lực bán tháo khiến thị trường giảm sàn hàng loạt
Tách từ: áp_lực bán_tháo khiến thị_trường giảm sàn hàng_loạt
AI Predict: NEG (0.99)
Manual Keywords: POS: [] | NEG: ['áp_lực', 'bán_tháo']
Manual Score: -0.50
==> FINAL HYBRID SCORE: -1.00
------------------------------
--- TEST SENTIMENT ---
Tiêu đề: Mặc dù gặp khó khăn, doanh nghiệp vẫn vượt kế hoạch lợi nhuận
Tách từ: mặc_dù gặp khó_khăn , doanh_nghiệp vẫn vượt kế_hoạch lợi_nhuận
AI Predict: POS (0.91)
Manual Keywords: POS: [] | NEG: []
Manual Score: +0.00
==> FINAL HYBRID SCORE: +0.91
------------------------------
--- TEST SENTIMENT ---
Tiêu đề: Cảnh báo nợ xấu tăng cao tại các ngân hàng
Tách từ: cảnh_báo nợ xấu tăng cao tại các

In [16]:
daily_summary = news_df.groupby(['date', 'ticker']).agg({
    'final_sentiment': ['mean', 'max', 'min', 'count']
}).reset_index()

# Làm phẳng cột (Flatten columns)
daily_summary.columns = ['date', 'ticker', 'sent_mean', 'sent_max', 'sent_min', 'news_count']

# Lưu file hoàn chỉnh để bắt đầu build RNN
daily_summary.to_csv('/content/drive/MyDrive/final_sentiment_features_v1.csv', index=False)